In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Load
df = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv')
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['resolution_bucket'].value_counts().sort_index())
print(f"\nColumns:")
print(df.columns.tolist())

c:\Users\Karnaveer Singh\Justice_Hq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (493776, 11)

Target distribution:
resolution_bucket
0_under_6months      158188
1_six_to_24months    185906
2_over_2years        149682
Name: count, dtype: int64

Columns:
['ddl_case_id', 'cino', 'state_code', 'dist_code', 'court_no', 'judge_position', 'type_name', 'filing_year', 'filing_quarter', 'resolution_days', 'resolution_bucket']


In [3]:
import pandas as pd

# Load full dataset for court-history features
df_full = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\raw\commercial_clean.csv')

# Convert dates
df_full['date_of_filing'] = pd.to_datetime(df_full['date_of_filing'], errors='coerce')
df_full['date_of_decision'] = pd.to_datetime(df_full['date_of_decision'], errors='coerce')

# Keep only resolved cases with valid positive resolution time
df_full_resolved = df_full[df_full['date_of_decision'].notna()].copy()
df_full_resolved['resolution_days'] = (
    df_full_resolved['date_of_decision'] - df_full_resolved['date_of_filing']
).dt.days
df_full_resolved = df_full_resolved[
    (df_full_resolved['resolution_days'] > 0) &
    (df_full_resolved['date_of_filing'].notna())
].copy()

df_full_resolved = df_full_resolved.sort_values(['court_no', 'date_of_filing']).reset_index(drop=True)
print(f'Full resolved rows: {len(df_full_resolved):,}')
print(df_full_resolved[['court_no', 'date_of_filing', 'date_of_decision', 'resolution_days']].head())

Full resolved rows: 2,203,465
   court_no date_of_filing date_of_decision  resolution_days
0         1     2010-01-01       2010-07-15              195
1         1     2010-01-01       2016-04-06             2287
2         1     2010-01-01       2010-11-16              319
3         1     2010-01-01       2010-01-11               10
4         1     2010-01-01       2011-04-30              484


In [7]:
# Build lagged court-history features and merge them onto df
df_base = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv')

court_history = df_full_resolved[['ddl_case_id', 'court_no', 'date_of_filing', 'resolution_days']].copy()
court_history = court_history.sort_values(['court_no', 'date_of_filing', 'ddl_case_id'])

court_history['court_historical_median_resolution'] = (
    court_history.groupby('court_no')['resolution_days']
    .transform(lambda s: s.shift().expanding().median())
)

court_history['pending_cases_count'] = court_history.groupby('court_no').cumcount()

global_median_resolution = court_history['resolution_days'].median()
court_history['court_historical_median_resolution'] = court_history['court_historical_median_resolution'].fillna(global_median_resolution)

court_history_features = court_history[['ddl_case_id', 'court_historical_median_resolution', 'pending_cases_count']].copy()
df = df_base.merge(court_history_features, on='ddl_case_id', how='left')
df['court_historical_median_resolution'] = df['court_historical_median_resolution'].fillna(global_median_resolution)
df['pending_cases_count'] = df['pending_cases_count'].fillna(0)

print(f'Base df shape: {df_base.shape}')
print(f'Enriched df shape: {df.shape}')
print(df[['court_historical_median_resolution', 'pending_cases_count']].head())

Base df shape: (493776, 11)
Enriched df shape: (493776, 13)
   court_historical_median_resolution  pending_cases_count
0                               700.5                 1270
1                               662.0                10633
2                               690.0                 1802
3                               670.5                 9936
4                               661.0                 4591


In [2]:
df.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,filing_year,filing_quarter,resolution_days,resolution_bucket
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,2010,1,412,1_six_to_24months
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,2010,4,565,1_six_to_24months
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,2010,1,846,2_over_2years
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,2010,4,744,2_over_2years
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,2010,2,207,1_six_to_24months


In [19]:
# Features and target
FEATURE_COLS = [
    'state_code', 'dist_code', 'court_no',
    'judge_position', 'type_name',
    'filing_year', 'filing_quarter',
    'court_historical_median_resolution', 'pending_cases_count'
]

# New target buckets: <1 year, 1-3 years, >3 years
df['resolution_bucket_v2'] = pd.cut(
    df['resolution_days'],
    bins=[-np.inf, 365, 1095, np.inf],
    labels=['0_under_1year', '1_one_to_3years', '2_over_3years']
)
TARGET = 'resolution_bucket_v2'

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()
valid_idx = y.notna()
X = X[valid_idx].copy()
y = y[valid_idx].copy()
print('New target distribution:')
print(y.value_counts().sort_index())

# Encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(f"Target classes: {le_target.classes_}")

# Encode all features
encoders = {}
categorical_cols = ['state_code', 'dist_code', 'court_no', 'judge_position', 'type_name']
numeric_cols = ['filing_year', 'filing_quarter', 'court_historical_median_resolution', 'pending_cases_count']

for col in categorical_cols:
    enc = LabelEncoder()
    X[col] = enc.fit_transform(X[col].astype(str))
    encoders[col] = enc

for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Train test split — 80/20, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_encoded
)

print(f"\nTrain size: {X_train.shape[0]:,}")
print(f"Test size:  {X_test.shape[0]:,}")
print(f"\nTrain class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(le_target.classes_, counts):
    print(f"  {cls}: {cnt:,} ({cnt/len(y_train)*100:.1f}%)")

New target distribution:
resolution_bucket_v2
0_under_1year      242993
1_one_to_3years    170104
2_over_3years       80679
Name: count, dtype: int64
Target classes: ['0_under_1year' '1_one_to_3years' '2_over_3years']

Train size: 395,020
Test size:  98,756

Train class distribution:
  0_under_1year: 194,394 (49.2%)
  1_one_to_3years: 136,083 (34.4%)
  2_over_3years: 64,543 (16.3%)


In [13]:
def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"Jindal benchmark: 81.40%")
    print(f"Difference: {(acc-0.814)*100:+.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, 
          target_names=le_target.classes_))
    return acc

In [5]:
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_acc = evaluate_model("XGBoost", y_test, xgb_preds)

Training XGBoost...

Model: XGBoost
Accuracy: 58.53%
Jindal benchmark: 81.40%
Difference: -22.87%

Classification Report:
                   precision    recall  f1-score   support

  0_under_6months       0.73      0.60      0.66     31638
1_six_to_24months       0.52      0.69      0.59     37181
    2_over_2years       0.56      0.45      0.50     29937

         accuracy                           0.59     98756
        macro avg       0.60      0.58      0.58     98756
     weighted avg       0.60      0.59      0.58     98756



In [6]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)
dummy_acc = evaluate_model("Dummy Baseline", y_test, dummy_preds)


Model: Dummy Baseline
Accuracy: 37.65%
Jindal benchmark: 81.40%
Difference: -43.75%

Classification Report:
                   precision    recall  f1-score   support

  0_under_6months       0.00      0.00      0.00     31638
1_six_to_24months       0.38      1.00      0.55     37181
    2_over_2years       0.00      0.00      0.00     29937

         accuracy                           0.38     98756
        macro avg       0.13      0.33      0.18     98756
     weighted avg       0.14      0.38      0.21     98756



In [7]:
feat_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feat_importance)

          feature  importance
4       type_name    0.372650
0      state_code    0.137563
5     filing_year    0.117923
2        court_no    0.101067
6  filing_quarter    0.092342
1       dist_code    0.091676
3  judge_position    0.086779


Type of case is the single biggest predictor of resolution time. That makes legal sense — a cheque bounce case (NI Act Section 138) has a defined fast-track process, while a title dispute can drag on for decades.
State matters more than district — judicial culture at the state level (Orissa vs Himachal Pradesh) dominates over individual district variation.
Filing year has real signal — courts did get faster over 2010-2015 as commercial courts were established.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

models = {
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=6, 
                                   learning_rate=0.1, random_state=42, 
                                   verbosity=0, eval_metric='mlogloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=300, max_depth=6, 
                                    learning_rate=0.1, random_state=42, 
                                    verbosity=-1),
    'CatBoost': CatBoostClassifier(iterations=300, depth=6, 
                                    learning_rate=0.1, random_state=42, 
                                    verbose=0),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=6, 
                                            random_state=42, n_jobs=-1)
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = evaluate_model(name, y_test, preds)
    results[name] = acc

print("\n=== FINAL COMPARISON ===")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:15} {acc*100:.2f}%")

Training XGBoost...

Model: XGBoost
Accuracy: 58.46%
Jindal benchmark: 81.40%
Difference: -22.94%

Classification Report:
                   precision    recall  f1-score   support

  0_under_6months       0.73      0.60      0.66     31638
1_six_to_24months       0.52      0.68      0.59     37181
    2_over_2years       0.56      0.45      0.50     29937

         accuracy                           0.58     98756
        macro avg       0.60      0.58      0.58     98756
     weighted avg       0.60      0.58      0.58     98756

Training LightGBM...

Model: LightGBM
Accuracy: 58.50%
Jindal benchmark: 81.40%
Difference: -22.90%

Classification Report:
                   precision    recall  f1-score   support

  0_under_6months       0.72      0.60      0.66     31638
1_six_to_24months       0.52      0.68      0.59     37181
    2_over_2years       0.56      0.45      0.50     29937

         accuracy                           0.58     98756
        macro avg       0.60      0.58   

In [20]:
# Run only one XGBoost model with the provided best params
best_xgb_params = {
    'subsample': 0.8,
    'n_estimators': 500,
    'min_child_weight': 5,
    'max_depth': 10,
    'learning_rate': 0.05,
    'colsample_bytree': 0.8
}

print(f'Using XGBoost params: {best_xgb_params}')

xgb_best_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=-1,
    **best_xgb_params
)

xgb_best_model.fit(X_train, y_train)
xgb_best_only_preds = xgb_best_model.predict(X_test)
xgb_best_only_acc = evaluate_model('XGBoost Best Params Only', y_test, xgb_best_only_preds)
print(f'Final accuracy: {xgb_best_only_acc*100:.2f}%')

Using XGBoost params: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Model: XGBoost Best Params Only
Accuracy: 61.23%
Jindal benchmark: 81.40%
Difference: -20.17%

Classification Report:
                 precision    recall  f1-score   support

  0_under_1year       0.67      0.74      0.71     48599
1_one_to_3years       0.52      0.58      0.55     34021
  2_over_3years       0.64      0.28      0.39     16136

       accuracy                           0.61     98756
      macro avg       0.61      0.54      0.55     98756
   weighted avg       0.62      0.61      0.60     98756

Final accuracy: 61.23%


In [18]:
# Confusion matrix
cm = confusion_matrix(y_test, xgb_best_only_preds)
cm_df = pd.DataFrame(
    cm,
    index=le_target.classes_,
    columns=['Pred: under_6m', 'Pred: 6_24m', 'Pred: over_2yr']
)
print('Confusion Matrix (Row=Actual, Col=Predicted):')
print(cm_df)

# Error analysis — what % of each class is misclassified as adjacent vs far
print('\nRow percentages (how each actual class gets predicted):')
cm_pct = cm_df.div(cm_df.sum(axis=1), axis=0).mul(100).round(1)
print(cm_pct)

Confusion Matrix (Row=Actual, Col=Predicted):
                   Pred: under_6m  Pred: 6_24m  Pred: over_2yr
0_under_6months             19738         8602            3298
1_six_to_24months            4529        25110            7542
2_over_2years                2803        12686           14448

Row percentages (how each actual class gets predicted):
                   Pred: under_6m  Pred: 6_24m  Pred: over_2yr
0_under_6months              62.4         27.2            10.4
1_six_to_24months            12.2         67.5            20.3
2_over_2years                 9.4         42.4            48.3
